In [ ]:
!pip install -U bitsandbytes transformers peft datasets huggingface_hub tqdm

In [1]:
import os
import re
import random
from functools import partial
from typing import Any

import numpy as np
import torch
from datasets import concatenate_datasets, load_dataset
from google.colab import userdata
from huggingface_hub import login
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [2]:
# Authentication + runtime checks
hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    hf_token = userdata.get("HF_TOKEN_WRITE")

if hf_token:
    login(token=hf_token)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA runtime not detected. Set Colab runtime to GPU.")

print(f"CUDA device: {torch.cuda.get_device_name(0)}")

CUDA device: NVIDIA A100-SXM4-40GB


In [3]:
# Configuration
SEED = 42
BASE_MODEL_ID = "google/medgemma-1.5-4b-it"
ADAPTER_ID = "gabrielbuzzi/medgemma-aci-lora"

PROMPT = """
System Prompt:
You are a medical scribe. Given a transcription of a doctor-patient consultation, generate a structured clinical note in the following format. Write in third person, past tense, using formal medical language. Synthesize and paraphrase — do not copy dialogue verbatim.
Required sections and guidance:
CHIEF COMPLAINT — One sentence stating the reason for the visit.
HISTORY OF PRESENT ILLNESS — Narrative paragraph(s) covering: patient demographics and relevant past medical history (PMH), then chronological account of current status for each active condition, lifestyle factors, medication adherence, and any symptoms endorsed or denied. Group related conditions together.
REVIEW OF SYSTEMS — Bulleted by organ system. Use "Endorses" or "Denies" for each relevant symptom. Only include systems discussed in the encounter.
PHYSICAL EXAMINATION — List findings by system. Only include what was explicitly examined.
VITALS REVIEWED — Summarize notable vitals only (e.g., "Elevated," "Within normal limits"). Omit if not discussed.
RESULTS — Summarize any lab, imaging, or diagnostic results referenced during the visit. Omit if none discussed.
ASSESSMENT AND PLAN — Restate patient demographics and PMH in the opening sentence. Then for each active diagnosis (in order of clinical priority), include relevant subsections chosen from: Medical Reasoning, Additional Testing, Medical Treatment, Patient Education and Counseling. Close with "Patient Agreements" confirming the patient's understanding.
Rules:

Only document what is present in the transcription; do not invent clinical details.
Use standard medical abbreviations and terminology (e.g., dyspnea, ideation, ejection fraction).
If a section has no relevant content from the transcript, omit it entirely.
Keep the tone objective and professional throughout.

Input: A raw transcription of a doctor-patient conversation.
Output: A complete clinical note following the format above.

Input:
{}

Output:
"""

TEST_NOISE_PROB = 0.0
TEST_SUBSET_SIZE = 256  # Set to 0 to run full test set
MAX_SEQ_LENGTH = 4096
MAX_NEW_TOKENS = 1024

OUTPUT_PREDICTIONS_PATH = "outputs/test_predictions_medgemma_lora.jsonl"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
# Dataset helpers
def clean_dialogue(text: str) -> str:
    text = re.sub(r"\[doctor\]", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\[patient\]", "", text, flags=re.IGNORECASE)
    text = text.replace("\\n", " ").replace("\n", " ")
    return re.sub(r"\s+", " ", text).strip()


def simulate_transcription_noise(text: str, drop_prob: float) -> str:
    words = text.split()
    words = [w for w in words if random.random() > drop_prob]
    return " ".join(words)


def format_test_example(prompt: str, example: dict[str, Any], noise_prob: float) -> dict[str, Any]:
    dialogue = clean_dialogue(example["dialogue"])
    if noise_prob > 0:
        dialogue = simulate_transcription_noise(dialogue, drop_prob=noise_prob)
    example["messages"] = [
        {"role": "user", "content": [{"type": "text", "text": prompt.format(dialogue)}]}
    ]
    return example

In [5]:
# Load test split
dataset = load_dataset("mkieffer/ACI-Bench", "aci")
test_ds = concatenate_datasets([dataset["test1"], dataset["test2"], dataset["test3"]])
test_ds = test_ds.map(partial(format_test_example, PROMPT, noise_prob=TEST_NOISE_PROB))
if TEST_SUBSET_SIZE > 0:
    test_ds = test_ds.select(range(min(TEST_SUBSET_SIZE, len(test_ds))))

print(test_ds)

Map:   0%|          | 0/66 [00:00<?, ? examples/s]

Dataset({
    features: ['encounter_id', 'dialogue', 'note', 'messages'],
    num_rows: 66
})


In [6]:
# Load tokenizer + base model + LoRA adapter
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=quantization_config,
    dtype=torch.bfloat16,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, ADAPTER_ID)
model.eval()

model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
if hasattr(model, "generation_config"):
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id

print("Model + adapter loaded.")

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/77.1M [00:00<?, ?B/s]

Model + adapter loaded.


In [16]:
# Inference
def generate_predictions(eval_dataset) -> list[str]:
    predictions = []
    for example in tqdm(eval_dataset, desc="Generating"):
        user_text = " ".join(
            " ".join(part["text"] for part in msg["content"])
            for msg in example["messages"]
            if msg["role"] == "user"
        )
        prompt_text = f"{user_text}{tokenizer.eos_token}"
        inputs = tokenizer(
            prompt_text,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            add_special_tokens=False,
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
        pred = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        predictions.append(pred)

    return predictions


predictions = generate_predictions([test_ds[1]])
references = test_ds["note"]

Generating:   0%|          | 0/1 [00:00<?, ?it/s]

In [15]:
print(f"Predictions: {len(predictions)}")
print("Sample prediction:\n")
print(predictions[0] if predictions else "(none)")

Predictions: 1
Sample prediction:

CHIEF COMPLAINT
Patient presents with new onset shortness of breath and wheezing.
HISTORY OF PRESENT ILLNESS
Patient is a 60-year-old male who presents with new onset shortness of breath and wheezing. Patient reports that he has been experiencing shortness of breath for the past two years since moving to the city. Patient reports that he has been experiencing mid shortness of breath suddenly for some reason for the past two years. Patient reports that he has been experiencing shortness of breath for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason for some reason 

In [ ]:
# Save predictions for offline evaluation
os.makedirs(os.path.dirname(OUTPUT_PREDICTIONS_PATH), exist_ok=True)
test_with_preds = test_ds.add_column("prediction", predictions)
test_with_preds.to_json(OUTPUT_PREDICTIONS_PATH, orient="records", lines=True)

print(f"Saved predictions to: {OUTPUT_PREDICTIONS_PATH}")